# demo

> Isolated rendering of markup with current styles and behaviors, for notebooks and docs

In [1]:
#| default_exp demo

The registry's `build_css` and `build_js` turn the styles and behaviors it holds into a stylesheet and a script. To develop a component, we need to see the result and try its interactions. `demo` renders ordinary FastHTML markup in an isolated document with the registry's current stylesheet and script. The same preview is used both during notebook authoring and when viewing generated docs.


In [2]:
from fastcore.utils import *
from fastcore.meta import delegates
from fasthtml.common import to_xml, Div, Iframe, Button, Html, Head, Body, Style, Script, FT
from fastcore.xml import highlight
from fasthtml.core import def_hdrs
from answerui.registry import reg, Hyperelement
import answerui.behaviors  # installs Registry.build_js
from fastlucide.icons import spritesheet

In [3]:
from answerui.registry import hypelt
from fastcore.test import *
from fasthtml.common import P, Details, Summary

## Rendering previews

A preview needs its own stylesheet and behavior script, the sprite sheet for icons, and htmx. FastHTML's `def_hdrs` returns the headers an app has by default: the charset and viewport meta tags, htmx 2, and FastHTML's helper script. `def_hdrs(htmx4=True)` returns htmx 4 instead, with an `htmx-config` meta tag that sets `metaCharacter` to `-`. `fast_app(htmx4=True)` gives an app the same headers. We pass `surreal=False` to leave out Surreal.

htmx comes before the behavior script in the head, because the script registers with htmx. Both load before the page's content. We can pass a stylesheet as `css` to try it in place of the registered one, without changing the registry. `demo_html` takes the stylesheet and script from `reg` unless another registry is passed as `registry`.

The preview's `body` is only as tall as its content, because an inline `min-block-size: auto` overrides any minimum height from an app's own `body` rule. Otherwise, an app that makes its page fill the viewport would make the frame grow without end.

When FastHTML serves a page, it renames attributes such as `get`, `post` and `link` to `hx-get`, `hx-post` and `href`, turning each value into a URL. Since nothing serves a preview, `demo_html` does the same renaming itself, but uses each value unchanged as the path. `routes` stands in for the server that would answer those requests: a table from `'GET path'` or `'POST path'` to the markup that answers it. A short script checks each htmx request before htmx sends it. When `routes` has markup for the request, the script cancels the request and swaps the markup into its target, with the swap style that the requesting element has or inherits. The preview then goes through the same states as the served page.


In [4]:
_verbs = dict(get='hx-get', post='hx-post', put='hx-put', delete='hx-delete', patch='hx-patch', link='href')
def _as_served(o):
    "`o` with `get`, `post`, `put`, `delete`, `patch` and `link` attributes renamed to `hx-get`, `hx-post`, `hx-put`, `hx-delete`, `hx-patch` and `href`, as when FastHTML serves a page"
    if isinstance(o, (list, tuple)): return [_as_served(c) for c in o]
    if not isinstance(o, FT): return o
    for k,v in _verbs.items():
        if k in o.attrs: o.attrs[v] = o.attrs.pop(k)
    for c in o.children: _as_served(c)
    return o

_routes_js = r'''const routes = %s;
document.addEventListener('htmx:beforeRequest', e => {
  const r = e.detail.requestConfig, html = routes[`${r.verb.toUpperCase()} ${r.path}`];
  if (html == null) return;
  e.preventDefault();
  const spec = (htmx.closest(e.detail.elt, '[hx-swap]')?.getAttribute('hx-swap') ?? 'innerHTML').split(' ')[0];
  htmx.swap(e.detail.target, html, {swapStyle: spec});
});'''

In [5]:
def demo_html(
    *fts,         # Markup to render
    registry=reg, # Registry whose stylesheet and behavior script the document carries
    css=None,     # Stylesheet for the document; None uses the registry's build_css()
    htmx4=False,  # Use htmx 4 with FastHTML's config, as `fast_app(htmx4=True)` does, instead of htmx 2
    routes=None,  # Canned responses for the document's htmx requests, `{'GET path': markup}`, standing in for a server
):
    "A standalone document containing markup, CSS, behaviors, the sprite sheet, and FastHTML's default headers"
    head = Head(*def_hdrs(htmx4=htmx4, surreal=False), Style(registry.build_css() if css is None else css), Script(registry.build_js()))
    if routes: head = head(Script(_routes_js % dumps({k: to_xml(_as_served(v)) for k,v in routes.items()})))
    return to_xml(Html(head, Body(*_as_served(fts), spritesheet, style='min-block-size: auto')))


In notebooks, `demo` shows the preview in an iframe. The notebook's styling, such as solveit's CSS, then can't affect the preview, and the preview's styling can't affect the notebook. The iframe's `srcdoc` attribute holds the complete document from `demo_html`. Unless `height` is given, a script in the document resizes the iframe to fit its contents. A replay button above the frame reloads the document, to restart dynamic components such as a notice that dismisses itself.

The frame holds the whole stylesheet and script, too much to store in a notebook with many previews. `demo` therefore returns a `Preview` with two representations: the frame as HTML, for the page, and the markup alone as Markdown, for the notebook file and for a model reading a dialog.

In [6]:
_sizer = Script(r"new ResizeObserver(() => frameElement.style.height = document.body.scrollHeight+24+'px').observe(document.body)")

class Preview:
    "A rendered preview: the frame as HTML for the page, the markup as Markdown for the notebook file and the model"
    def __init__(self, fts, frame): store_attr()
    def __repr__(self): return to_xml(self.fts)
    def _repr_mimebundle_(self, include=None, exclude=None): return {'text/html': to_xml(self.frame), 'text/markdown': highlight(self.fts)}, {'nbdev': {'strip': ['text/html']}}

def demo(
    *fts,         # Markup to render
    registry=reg, # Registry whose stylesheet and behavior script the preview carries
    css=None,     # Stylesheet for the document; None uses the registry's build_css()
    height=None,  # Fixed iframe height, e.g. '10rem', for top-layer content the frame can't measure; None follows the content
    width=None,   # Fixed iframe width, e.g. '24rem', to preview a narrow viewport; None fills the cell
    htmx4=False,  # Use htmx 4 with FastHTML's config, as `fast_app(htmx4=True)` does, instead of htmx 2
    routes=None,  # Canned responses for the preview's htmx requests, `{'GET path': markup}`, standing in for a server
):
    "Render markup in an isolated, replayable notebook preview"
    doc = fts if height is not None else fts + (_sizer,)
    style = f'width:100%;box-sizing:border-box;border:1px solid #e3e3e8;border-radius:8px;height:{height or "auto"};'
    frame = Iframe(srcdoc=demo_html(*doc, registry=registry, css=css, htmx4=htmx4, routes=routes), title='answerui preview', style=style)
    replay = Button('↻', type='button', aria_label='replay', title='replay',
        onclick=r"const f=this.parentElement.nextElementSibling; f.srcdoc=f.srcdoc",
        style='border:0;background:none;font-size:14px;opacity:.6;cursor:pointer;padding:0 4px')
    return Preview(fts, Div(Div(replay, style='text-align:right'), frame, style=f'width:{width or "100%"}'))


A preview does not require a registered component. Here `css=''` replaces the registry's stylesheet with an empty one, showing a native element exactly as the browser draws it:

In [7]:
demo(Details(Summary('More information'), P('This disclosure is a native details element.')), css='')

<details><summary>More information</summary>  <p>This disclosure is a native details element.</p>
</details>

Let's recreate the notice from the earlier notebooks. One `hypelt` call registers its styles and behaviors together. A click rule on its `dismiss` button removes the notice. With `autodismiss`, an `init` rule removes the notice that many milliseconds after htmx loads it, or 4000 by default.

In [8]:
notice_css = r"""
display: block;
padding: .75rem 1rem;
background: #f4f4f6;
border: 1px solid #ccc;
border-radius: 8px;
"""
NoticeBox = hypelt('NoticeBox', css=notice_css, layer='components', on={
    ('& [dismiss]', 'click'): r"el.closest('notice-box').remove()",
    ('&[autodismiss]', 'init'): r"await new Promise(r => setTimeout(r, +el.getAttribute('autodismiss') || 4000)); el.remove()"},
    doc='A dismissible notice.')

Click Dismiss, then the ↻ icon to restore the notice:

In [9]:
demo(NoticeBox('Changes saved. ', Button('Dismiss', type='button', dismiss=True)))

<notice-box>Changes saved. <button type="button" dismiss>Dismiss</button></notice-box>

`_repr_mimebundle_` returns the two representations, with metadata that marks the HTML with nbdev's `strip` flag. That flag tells `nbdev-clean` to drop the frame from the stored output and keep the Markdown:


In [10]:
p = demo(NoticeBox('Changes saved.'))
bundle, meta = p._repr_mimebundle_()
test_eq(set(bundle), {'text/html', 'text/markdown'})
assert 'srcdoc' in bundle['text/html']
assert 'srcdoc' not in bundle['text/markdown']
test_eq(meta, {'nbdev': {'strip': ['text/html']}})
print(bundle['text/markdown'])

```html
<notice-box>Changes saved.</notice-box>
```


This notice disappears after four seconds:

In [11]:
demo(NoticeBox('This notice disappears after four seconds.', autodismiss=4000), height='4rem')

<notice-box autodismiss="4000">This notice disappears after four seconds.</notice-box>

Here the button requests `/saved`, and the notice that `routes` gives for that request replaces the button. Click it, then ↻ to start again.

In [12]:
demo(Button('Save', type='button', get='/saved', hx_swap='outerHTML'), routes={'GET /saved': NoticeBox('Saved.')})

<button type="button" hx-swap="outerHTML" hx-get="/saved">Save</button>

The document contains the renamed `hx-get` attribute and the script with the `routes` table. htmx sends any request that the table has no answer for as usual.

In [13]:
h = demo_html(Button('Save', type='button', get='/saved'), routes={'GET /saved': NoticeBox('Saved.')})
assert 'hx-get="/saved"' in h and 'GET /saved' in h and 'htmx:beforeRequest' in h

`htmx4=True` gives a preview the headers of an app built with `fast_app(htmx4=True)`: htmx 4 with FastHTML's config. The behavior runtime runs `init` rules through `htmx.onLoad`. Under that config, `htmx.onLoad` listens for `htmx-after-process`, htmx 4's event for newly arrived content. The notice's timer still starts:

In [14]:
h = demo_html(NoticeBox('x'), htmx4=True)
assert 'htmx.org@4' in h and 'metaCharacter' in h
demo(NoticeBox('This notice disappears after four seconds under htmx 4.', autodismiss=4000), htmx4=True, height='4rem')

<notice-box autodismiss="4000">This notice disappears after four seconds under htmx 4.</notice-box>

Here we try a candidate stylesheet as `css`: the notice's CSS, with a pale yellow background in place of the grey. The registered behaviors still apply.

In [15]:
candidate_css = NoticeBox.rules['base'].compiled_css.replace('#f4f4f6', '#fff4cc')
demo(NoticeBox('Trying a warmer surface. ', Button('Dismiss', type='button', dismiss=True)), css=candidate_css)

<notice-box>Trying a warmer surface. <button type="button" dismiss>Dismiss</button></notice-box>

An application override is registered, not passed as `css`. `NoticeBox.rule` adds CSS to the `app` layer. Every later preview and application page includes this CSS. The rule leaves the notice's definition and behaviors unchanged:

In [16]:
NoticeBox.rule('corners', 'border-radius: 1rem;', doc='Rounder notice corners.')
demo(NoticeBox('The current stylesheet includes the application override.'))


<notice-box>The current stylesheet includes the application override.</notice-box>

## Reusable examples

A component can provide an example for someone exploring the library. The `example` decorator attaches a no-argument markup function to the hyperelement as `NoticeBox.example`. `NoticeBox.demo()` renders the function's markup with the component's registry.

Attaching another example replaces the previous one. To show several variants, return them together from one function.

In [17]:
def example(h):
    "Decorator that stores a no-argument markup function as `h.example`, for `h.demo()` to render"
    def _attach(f):
        h.example = f
        return f
    return _attach

@patch
@delegates(demo, but=['registry'])
def demo(self:Hyperelement, **kwargs):
    "Render this hyperelement's example with its registry's current styles and behaviors"
    if not hasattr(self, 'example'): raise ValueError(f'{self.__name__} has no example; attach one with @example({self.__name__})')
    return demo(self.example(), registry=self.registry, **kwargs)


In [18]:
@example(NoticeBox)
def notice_example():
    return Div(NoticeBox('Changes saved. ', Button('Dismiss', type='button', dismiss=True)),
        NoticeBox('This notice stays visible.'), style='display:grid;gap:1rem')

The example itself is ordinary markup:

In [19]:
to_xml(NoticeBox.example())

'<div style="display:grid;gap:1rem">\n<notice-box>Changes saved. <button type="button" dismiss>Dismiss</button></notice-box><notice-box>This notice stays visible.</notice-box></div>\n'

`NoticeBox.demo()` renders it with the current styles and behaviors:

In [20]:
NoticeBox.demo()


<div style="display:grid;gap:1rem">
<notice-box>Changes saved. <button type="button" dismiss>Dismiss</button></notice-box><notice-box>This notice stays visible.</notice-box></div>

`NoticeBox.example()` returns fresh FastHTML markup that can be inspected or reused in a larger composition. `NoticeBox.demo()` takes `demo`'s options, such as `height` and `css`. Each call to `NoticeBox.demo()` builds a new document from the current example, styles, and behaviors. The replay button instead restarts the preview's interactions by reloading the document already in the frame, without running Python.

Because each shipped component's notebook exports its `@example` function, importing the component's module attaches the example without rendering anything.


Registering `corners` again replaces the earlier rule in place. The next `NoticeBox.demo()` builds its document with the square corners:

In [21]:
NoticeBox.rule('corners', 'border-radius: 0;', doc='Square notice corners.')
NoticeBox.demo()

<div style="display:grid;gap:1rem">
<notice-box>Changes saved. <button type="button" dismiss>Dismiss</button></notice-box><notice-box>This notice stays visible.</notice-box></div>

Calling `hypelt` again with the same name updates the existing `NoticeBox` in place, keeping its example:

In [22]:
NoticeBox = hypelt('NoticeBox', css=notice_css, layer='components', doc='A persistent notice.')
test_is(NoticeBox.example, notice_example)
NoticeBox.demo()

<div style="display:grid;gap:1rem">
<notice-box>Changes saved. <button type="button" dismiss>Dismiss</button></notice-box><notice-box>This notice stays visible.</notice-box></div>

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()